<a href="https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data used for the feature check

This assignment uses the historical query-performance data from the FlyRank internship warehouse. The feature check focuses on the historical previous-30-day signals used by the capstone ranking workflow. The data is used only to construct and audit the predictive feature vector; client and content identifiers are not used as predictive inputs.

In [9]:
# ML-05 — Section 0: Load the query-performance data

from datasets import load_dataset
from itertools import islice
import pandas as pd
import numpy as np

# Load the same warehouse table used in the capstone workflow.
# Streaming keeps the notebook from loading the full warehouse into memory.

query_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    streaming=True
)

query_rows = list(islice(query_ds, 100000))

query_df = pd.DataFrame(query_rows)

print("Source table: fact_content_query_90d")
print("Rows loaded:", len(query_df))
print("Columns:", len(query_df.columns))

print("\nAvailable columns:")
print(query_df.columns.tolist())

print("\nWindow start:")
print(
    query_df["window_start"].min(),
    "to",
    query_df["window_start"].max()
)

print("\nWindow end:")
print(
    query_df["window_end"].min(),
    "to",
    query_df["window_end"].max()
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Source table: fact_content_query_90d
Rows loaded: 100000
Columns: 21

Available columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

Window start:
2026-04-02 to 2026-04-02

Window end:
2026-06-30 to 2026-06-30


## 1. Build the feature vector

The feature vector contains five historical search-performance signals used to rank content for refresh review: previous-30-day impressions, previous-30-day clicks, previous-30-day CTR, visible query count, and rare query count. These features describe information available before the outcome being evaluated. The selected inputs are numeric, so no categorical encoding is required. Missing numeric values are handled with median filling during feature preparation.

In [10]:
# ML-05 — Section 1: Build the feature vector

# Rename the source field to the modeling feature name used in the capstone.
query_df["visible_query_count"] = query_df["content_visible_query_count"]

feature_cols = [
    "impressions_prev30",
    "clicks_prev30",
    "visible_query_count",
    "rare_query_count",
]

# CTR is derived only from the previous-30-day historical window.
query_df["ctr_prev30"] = np.where(
    query_df["impressions_prev30"] > 0,
    query_df["clicks_prev30"] / query_df["impressions_prev30"],
    0
)

feature_cols.append("ctr_prev30")

# Build the actual feature vector.
X = query_df[feature_cols].copy()

# Ensure numeric representation.
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Fill missing numeric values with the median.
for col in feature_cols:
    X[col] = X[col].fillna(X[col].median())

print("Final feature vector:")
for col in feature_cols:
    print("-", col)

print("\nFeature matrix shape:", X.shape)

print("\nMissing values after preparation:")
print(X.isna().sum())

print("\nFeature preview:")
display(X.head())

Final feature vector:
- impressions_prev30
- clicks_prev30
- visible_query_count
- rare_query_count
- ctr_prev30

Feature matrix shape: (100000, 5)

Missing values after preparation:
impressions_prev30     0
clicks_prev30          0
visible_query_count    0
rare_query_count       0
ctr_prev30             0
dtype: int64

Feature preview:


,impressions_prev30,clicks_prev30,visible_query_count,rare_query_count,ctr_prev30
0,11,0,14,32,0.0
1,1,0,14,32,0.0
2,5,0,14,32,0.0
3,1,0,14,32,0.0
4,0,0,14,32,0.0


### Feature notes

- **impressions_prev30** — impressions observed during the previous 30-day window. Missing values are median-filled. This is historical information available before the evaluated outcome.
- **clicks_prev30** — clicks observed during the previous 30-day window. Missing values are median-filled. This is historical information available before the evaluated outcome.
- **ctr_prev30** — click-through rate calculated from the previous 30-day window. Missing values are median-filled. It represents historical search performance available before the outcome.
- **visible_query_count** — number of visible queries associated with the content item in the historical query data. Missing values are median-filled.
- **rare_query_count** — count of rare queries associated with the content item in the historical query data. Missing values are median-filled.

All five predictive inputs are numeric. No categorical encoding is required. The features are intended to represent information available before the outcome window rather than information from the future evaluation period.

In [11]:
# ML-05 — Section 2: Feature notes

feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "meaning": [
        "Impressions during the previous 30-day window",
        "Clicks during the previous 30-day window",
        "CTR calculated from previous-30-day clicks and impressions",
        "Number of visible queries associated with the content",
        "Number of rare queries associated with the content",
    ],
    "type": [
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
    ],
    "missing_handling": [
        "Median fill",
        "Median fill",
        "Median fill",
        "Median fill",
        "Median fill",
    ],
    "available_before_outcome": [
        True,
        True,
        True,
        True,
        True,
    ],
})

display(feature_notes)

print("\nFeature dtypes:")
print(X.dtypes)

print("\nMissing values:")
print(X.isna().sum())

,feature,meaning,type,missing_handling,available_before_outcome
0,impressions_prev30,Impressions during the previous 30-day window,numeric,Median fill,True
1,clicks_prev30,Clicks during the previous 30-day window,numeric,Median fill,True
2,visible_query_count,CTR calculated from previous-30-day clicks and...,numeric,Median fill,True
3,rare_query_count,Number of visible queries associated with the ...,numeric,Median fill,True
4,ctr_prev30,Number of rare queries associated with the con...,numeric,Median fill,True



Feature dtypes:
impressions_prev30       int64
clicks_prev30            int64
visible_query_count      int64
rare_query_count         int64
ctr_prev30             float64
dtype: object

Missing values:
impressions_prev30     0
clicks_prev30          0
visible_query_count    0
rare_query_count       0
ctr_prev30             0
dtype: int64


## 3. The leakage hunt

The main leakage risks are fields that directly describe the outcome, fields calculated from a future window, and identifiers or client-specific fields that do not represent generalizable search-performance information. The target-derived field `trend_direction` is not used as a predictive feature. Pseudonymous identifiers such as `client_hash_id` and `content_hash_id` are excluded from the feature vector. Future-window outcome fields are also excluded because they would reveal information that occurs after the prediction point.

In [12]:
# ML-05 — Section 3: Leakage hunt

leakage_candidates = [
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "impressions_last30",
    "clicks_last30",
    "trend_direction",
    "trend_pct",
    "target",
    "future_window",
    "future_impressions",
    "future_clicks",
    "future_ctr",
]

available_columns = set(query_df.columns)

leakage_check = pd.DataFrame({
    "candidate": leakage_candidates,
    "present_in_source": [
        col in available_columns
        for col in leakage_candidates
    ],
    "used_as_feature": [
        col in feature_cols
        for col in leakage_candidates
    ],
})

display(leakage_check)

used_leakage = [
    col for col in leakage_candidates
    if col in feature_cols
]

print("\nLeakage candidates used as predictive features:")

if used_leakage:
    print(used_leakage)
else:
    print("None")

print("\nFinal predictive features:")
for col in feature_cols:
    print("-", col)

,candidate,present_in_source,used_as_feature
0,client_hash_id,True,False
1,content_hash_id,True,False
2,query_hash_id,True,False
3,impressions_last30,True,False
4,clicks_last30,True,False
5,trend_direction,False,False
6,trend_pct,False,False
7,target,False,False
8,future_window,False,False
9,future_impressions,False,False



Leakage candidates used as predictive features:
None

Final predictive features:
- impressions_prev30
- clicks_prev30
- visible_query_count
- rare_query_count
- ctr_prev30


### Fields deliberately excluded

- **client_hash_id** — pseudonymous client identifier; it is not used as a predictive feature.
- **content_hash_id** — pseudonymous content identifier; it is used only for identification/grouping, not prediction.
- **query_hash_id** — pseudonymous query identifier; it is not used as a predictive feature.
- **impressions_last30** — represents the more recent window and is excluded from the historical feature vector to keep the selected inputs aligned with the pre-outcome feature definition.
- **clicks_last30** — excluded for the same reason as `impressions_last30`.
- **trend_direction** — target/outcome information and therefore cannot be used as a predictive input.
- **trend_pct** — target-derived movement information and therefore excluded.
- **target** — the evaluation label and therefore excluded from the feature matrix.
- **Future-window outcome fields** — excluded because they would contain information from after the prediction point.
- **Product/client-specific flags** — excluded because they may encode entity-specific information rather than generalizable search-performance signals.

In [13]:
# ML-05 — Section 4: What I excluded and why

excluded_fields = pd.DataFrame({
    "field": [
        "client_hash_id",
        "content_hash_id",
        "query_hash_id",
        "impressions_last30",
        "clicks_last30",
        "trend_direction",
        "trend_pct",
        "target",
        "Future-window outcome fields",
        "Product/client-specific flags",
    ],
    "reason": [
        "Pseudonymous client identifier; not a predictive feature.",
        "Pseudonymous content identifier; not a predictive feature.",
        "Pseudonymous query identifier; not a predictive feature.",
        "Excluded from the selected historical feature vector.",
        "Excluded from the selected historical feature vector.",
        "Target/outcome information; would cause leakage.",
        "Target-derived movement information; would cause leakage.",
        "Evaluation label; cannot be used as an input.",
        "Contains information from after the prediction point.",
        "May encode entity-specific information rather than generalizable signals.",
    ],
})

display(excluded_fields)

print("Predictive feature count:", len(feature_cols))

print("\nFinal predictive feature set:")
for feature in feature_cols:
    print("-", feature)

,field,reason
0,client_hash_id,Pseudonymous client identifier; not a predicti...
1,content_hash_id,Pseudonymous content identifier; not a predict...
2,query_hash_id,Pseudonymous query identifier; not a predictiv...
3,impressions_last30,Excluded from the selected historical feature ...
4,clicks_last30,Excluded from the selected historical feature ...
5,trend_direction,Target/outcome information; would cause leakage.
6,trend_pct,Target-derived movement information; would cau...
7,target,Evaluation label; cannot be used as an input.
8,Future-window outcome fields,Contains information from after the prediction...
9,Product/client-specific flags,May encode entity-specific information rather ...


Predictive feature count: 5

Final predictive feature set:
- impressions_prev30
- clicks_prev30
- visible_query_count
- rare_query_count
- ctr_prev30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.